# E30 — phan hoach code goc, KHONG selector

Doi **`JOB`** o Cell 2 roi **Run All** (Save Version). Moi lan chay = 1 job.

## Vi sao co notebook nay

Nhanh cu cat `r_heldout` (10 % cua D_r) va `sel` (25 % cua D_t) de nuoi selector S_val.
S_val con nuoi `ReduceLROnPlateau`, tuc no **doi ca learning rate** chu khong chi chon
checkpoint. Ban E30 bo han hai lat cat do:

| | nhanh cu | E30 |
|---|---|---|
| `retain` | 5 410 (90 % D_r) | **6 010** (toan bo D_r) |
| tap bao cao | 398 (75 % D_t) | **531** (toan bo D_t) |
| `r_heldout` / `sel` | 600 / 133 | khong ton tai |
| learning rate | giam theo S_val | hang so sau warmup |

Dung y cach `forgetmi_partial.py` ban tac gia dung du lieu.

## 18 job

| nhom | JOB | so gio |
|---|---|---:|
| **De xuat 1** (P3-NoKD-More) | `p3_m3` `p3_m6` `p3_m10` `p3_iu` | ~0,6 / 1,1 / 1,8 / 0,6 |
| **Forget-MI** baseline | `fmi_m3` `fmi_m6` `fmi_m10` `fmi_iu` | ~0,8 / 1,4 / 2,2 / 0,7 |
| **NegGrad+** | `ng_m3` `ng_m6` `ng_m10` | chua do |
| **CF-k** | `cf_m3` `cf_m6` `cf_m10` | chua do |
| **OG + GOLD** (chi do lai) | `ref_m3` `ref_m6` `ref_m10` `ref_iu` | ~0,2 moi job |

`ref_*` KHONG huan luyen — OG va GOLD la checkpoint co san, chi do lai tren tap test 531.

## Chay job nao truoc

Chay **`p3_m3`** mot minh truoc. Xem log co du ba dong nay khong:

    [split-e30] retain=6010 ... test_final=531 ... khong cat r_heldout/sel
    adv_e30: PHAN HOACH THEO CODE GOC
    skip_selection=1 -> BO S_val moi epoch

Du ba dong moi chay tiep 17 job con lai.

## Input can Add

* MIMIC: `forget-mi-data` + `forget-mi-models-full`
* IU: `forget-mi-data-iu` + `forget-mi-models-iu` (+ `forget-mi-models-iu-re`) + `chest-xrays-indiana-university`

## Ket qua ghi vao dau

    /kaggle/working/results_e30_p3.csv          <- de xuat 1
    /kaggle/working/results_e30_forgetmi.csv    <- Forget-MI
    /kaggle/working/results_e30_baselines.csv   <- NegGrad+ / CF-k
    /kaggle/working/results_e30_reference.csv   <- OG / GOLD

Moi job ghi them mot hang, chay nhieu job thi file tu day len. **Chi doc hang
`checkpoint_kind == 'last'`** — hang `val_best` (neu co) chon tren tap val nam trong
retain, day la loi cua code goc duoc tai lap nguyen ven, khong dung de bao cao.


In [ ]:
# Cell 1: SETUP — clone repo + deps + chot chan code E30 da push
import os, subprocess

WORK = '/kaggle/working'
REPO = f'{WORK}/Forget-MI-LoKU'
REPO_URL = 'https://github.com/nhnhu146/Forget-MI-LoKU.git'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
os.chdir(REPO)

NEED = ['training/adv_e30.py', 'training/forgetmi_e30.py', 'training/forgetmi_partial_e30.py',
        'training/adv_common.py', 'training/forgetmi_p3_cand.py', 'training/forgetmi_partial.py',
        'training/forgetmi_eval_only.py', 'scripts/unlearn_baselines.py']
_miss = [f for f in NEED if not os.path.exists(f)]
assert not _miss, f'THIEU {_miss} -> git push code E30 roi Import lai notebook'

_e = open('training/adv_e30.py', encoding='utf-8').read()
assert 'data_split_original' in _e and '_ORIG_SETUP_EXPERIMENT' in _e, \
    'adv_e30.py la ban cu -> git push ban moi nhat'
_l = open('training/forgetmi_e30.py', encoding='utf-8').read()
assert 'C.build_dataset = E.build_dataset_e30' in _l, \
    'forgetmi_e30.py thieu hoan build_dataset -> git push ban moi nhat'
print('OK: 3 file E30 day du va dung phien ban.')

subprocess.run(['pip', 'install', '-q', 'pydicom', 'scikit-image', 'scikit-learn',
                'pyyaml', 'wandb', 'seaborn==0.13.2'], check=True)
subprocess.run(['pip', 'install', '-q', 'transformers==4.38.0', 'peft==0.10.0',
                'accelerate==0.27.0'], check=True)

import torch
assert torch.cuda.is_available(), 'Bat GPU trong Kaggle Settings truoc khi chay'
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('GPU   :', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2: CHON JOB + tim duong dan
import glob, os

# ============================ DOI DUNG DONG NAY ============================
JOB = 'p3_m3'
# ===========================================================================
#  De xuat 1 : p3_m3   p3_m6   p3_m10   p3_iu
#  Forget-MI : fmi_m3  fmi_m6  fmi_m10  fmi_iu
#  NegGrad+  : ng_m3   ng_m6   ng_m10
#  CF-k      : cf_m3   cf_m6   cf_m10
#  OG + GOLD : ref_m3  ref_m6  ref_m10  ref_iu     (khong huan luyen, chi do lai)

SEED   = 42
EPOCHS = 30
CFG    = 'f2'      # cau hinh cua De xuat 1 — chi co tac dung voi job p3_*

JOBS = {
 'p3_m3':  ('p3',  'mimic', 3),  'p3_m6':  ('p3',  'mimic', 6),
 'p3_m10': ('p3',  'mimic', 10), 'p3_iu':  ('p3',  'iu',    3),
 'fmi_m3': ('fmi', 'mimic', 3),  'fmi_m6': ('fmi', 'mimic', 6),
 'fmi_m10':('fmi', 'mimic', 10), 'fmi_iu': ('fmi', 'iu',    3),
 'ng_m3':  ('ng',  'mimic', 3),  'ng_m6':  ('ng',  'mimic', 6),  'ng_m10': ('ng', 'mimic', 10),
 'cf_m3':  ('cf',  'mimic', 3),  'cf_m6':  ('cf',  'mimic', 6),  'cf_m10': ('cf', 'mimic', 10),
 'ref_m3': ('ref', 'mimic', 3),  'ref_m6': ('ref', 'mimic', 6),
 'ref_m10':('ref', 'mimic', 10), 'ref_iu': ('ref', 'iu',    3),
}
assert JOB in JOBS, f'JOB phai thuoc {sorted(JOBS)}'
METHOD, DATASET, PCT = JOBS[JOB]


def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'):
            return f'/kaggle/input/{s}'
        h = glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h:
            return sorted(h)[0]
    return None


def bins(root):
    return sorted(glob.glob(os.path.join(root, '**', 'pytorch_model.bin'), recursive=True), key=len)


if DATASET == 'mimic':
    DATA = fd('forget-mi-data')
    MOD  = fd('forget-mi-models-full', 'forget-mi-models')
    assert DATA and MOD, 'Add Input: forget-mi-data + forget-mi-models-full'
    BASE = os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    _gh  = [b for b in bins(MOD) if f'model_retrained_{PCT}per' in b]
    # KHONG lui ve BASE khi thieu: GOLD am tham thanh OG thi moi so 'so voi gold' deu sai.
    assert _gh, f'Thieu model_retrained_{PCT}per trong {MOD}'
    GOLD   = os.path.dirname(_gh[0])
    TEXT   = os.path.join(DATA, 'data', 'metadata')
    IMG    = os.path.join(DATA, 'data', 'img_data')
    SPLIT  = './data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
    FORGET = f'./data_splits/forget_set_{PCT}per.csv'
    CFG_P3, CFG_BASE = 'config_advanced_kaggle.yaml', 'config_baseline_kaggle.yaml'
else:
    DATA  = fd('forget-mi-data-iu')
    MOD   = fd('forget-mi-models-iu')
    MODRE = fd('forget-mi-models-iu-re')
    RAD   = fd('chest-xrays-indiana-university')
    assert DATA and MOD and RAD, 'Add Input: forget-mi-data-iu + forget-mi-models-iu(+ -re) + raddar'
    _ogb = [b for b in bins(MOD) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD)
    BASE = os.path.dirname(_ogb[0])
    _reb = (bins(MODRE) if MODRE else []) or [b for b in bins(MOD) if 'retrain' in b.lower()]
    assert _reb, 'Thieu checkpoint retrained cho IU'
    GOLD   = os.path.dirname(_reb[0])
    _tsv   = (glob.glob(os.path.join(DATA, '**', 'all_data.tsv'), recursive=True)
              or glob.glob('/kaggle/input/**/all_data.tsv', recursive=True))
    TEXT   = os.path.dirname(_tsv[0])
    IMG    = (glob.glob(os.path.join(RAD, '**', 'images_normalized'), recursive=True) or [RAD])[0]
    SPLIT  = (glob.glob(os.path.join(DATA, '**', 'iu-split.csv'), recursive=True)
              + glob.glob('/kaggle/input/**/iu-split.csv', recursive=True))[0]
    FORGET = (glob.glob(os.path.join(DATA, '**', f'forget_set_{PCT}per_iu.csv'), recursive=True)
              + glob.glob(f'/kaggle/input/**/forget_set_{PCT}per_iu.csv', recursive=True))[0]
    CFG_P3, CFG_BASE = 'config_loku_iu_kaggle.yaml', 'config_baseline_iu_kaggle.yaml'

for _n, _p in {'BASE': BASE, 'GOLD': GOLD, 'TEXT': TEXT,
               'IMG': IMG, 'SPLIT': SPLIT, 'FORGET': FORGET}.items():
    assert _p and os.path.exists(_p), f'Missing {_n}: {_p}'

RID = f'{JOB}_e30_s{SEED}'
OD  = f'/kaggle/working/out_{RID}'

# Moi nhom mot file ket qua -> chay nhieu job thi file tu day len, de gop bang.
RESULTS = {'p3':  '/kaggle/working/results_e30_p3.csv',
           'fmi': '/kaggle/working/results_e30_forgetmi.csv',
           'ng':  '/kaggle/working/results_e30_baselines.csv',
           'cf':  '/kaggle/working/results_e30_baselines.csv',
           'ref': '/kaggle/working/results_e30_reference.csv'}[METHOD]

COMMON = {'forget_set_path': FORGET, 'base_model_path': BASE, 'bert_pretrained_dir': BASE,
          'retrained_model_path': GOLD, 'text_data_dir': TEXT, 'img_data_dir': IMG,
          'data_split_path': SPLIT, 'use_noise': 1,
          'output_dir': OD, 'results_csv_path': RESULTS, 'id': RID}

# ---- cau hinh rieng cua De xuat 1 (P3-NoKD-More, scheme uni_nokd) ----
# Chi khac nhau o lora_image_last_k_blocks: 3 (f6) vs 2 (f2).
CONFIGS = {
 'f6': {'lambda_ihl': 5.0, 'lambda_ce': 0.25, 'loku_subtract_scale': 1.0,
        'loku_image_subtract_scale': 1.0, 'lora_image_last_k_blocks': 3,
        'lora_image_include_fc1': 0},
 'f2': {'lambda_ihl': 5.0, 'lambda_ce': 0.25, 'loku_subtract_scale': 1.0,
        'loku_image_subtract_scale': 1.0, 'lora_image_last_k_blocks': 2,
        'lora_image_include_fc1': 0},
}
assert CFG in CONFIGS, f'CFG phai thuoc {sorted(CONFIGS)}'
OVR_P3 = dict(CONFIGS[CFG])
OVR_P3['lora_extra_target_modules'] = 'attention.output.dense|intermediate.dense|output.dense'
OVR_P3['unlearn_epochs'] = EPOCHS
OVR_P3['history_csv_path'] = f'/kaggle/working/perepoch_{RID}.csv'
OVR_P3['ce_selector'] = 0
if DATASET == 'iu':
    # config_loku_iu dung lr 5e-4 + clip 1.0; khoi phuc dung cau hinh da chot tu MIMIC.
    OVR_P3.update({'learning_rate': 2.0e-4, 'grad_clip': 0.0})

CKPT = f'{OD}/checkpoints/latest.pt'   # p3
EXP_PARAMS = {'f6': 1488896, 'f2': 1451008}[CFG]

print('=' * 72)
print(f'JOB      : {JOB}   ({METHOD} | {DATASET} | quen {PCT}%)')
print(f'run id   : {RID}')
print(f'config   : {CFG_P3 if METHOD in ("p3", "ref") else CFG_BASE}')
print(f'ket qua  : {RESULTS}')
print(f'output   : {OD}')
if METHOD == 'p3':
    print(f'CFG      : {CFG}  (cho doi {EXP_PARAMS:,} tham so)')
if METHOD == 'ref':
    print('CHI DO LAI OG + GOLD tren test = 531 — khong huan luyen')
print('=' * 72)

In [ ]:
# Cell 3: CHAY
import os, subprocess, time

env = {**os.environ, 'PYTHONPATH': '.', 'WANDB_MODE': 'disabled',
       'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}


def sh(cmd, what):
    print('=' * 72 + f'\n{what}\n' + '=' * 72)
    print(' '.join(cmd[:4]), '...')
    t0 = time.time()
    try:
        subprocess.run(cmd, env=env, check=True)
        print(f'OK {what}   {(time.time() - t0) / 3600:.2f}h')
        return True
    except subprocess.CalledProcessError as e:
        print(f'FAIL {what}  rc={e.returncode}')
        return False


def arg(d):
    return ','.join(f'{k}={v}' for k, v in d.items())


if METHOD == 'p3':
    # De xuat 1 — launcher hoan setup_experiment/build_dataset sang ban E30,
    # driver forgetmi_p3_cand.py giu nguyen khong sua mot dong.
    ovr = dict(COMMON); ovr.update(OVR_P3)
    sh(['python', 'training/forgetmi_e30.py', 'training/forgetmi_p3_cand.py',
        '--config', CFG_P3, '--seed', str(SEED),
        '--scheme', 'uni_nokd', '--ablate', 'none', '--fresh',
        '--override', arg(ovr)], f'TRAIN {RID}')

elif METHOD == 'fmi':
    # Forget-MI baseline — launcher rieng vi no dung split cua forgetmi_partial,
    # khong di qua adv_common.
    ovr = dict(COMMON)
    ovr.update({'unlearn_epochs': EPOCHS, 'evaluate_last_and_best': 1})
    sh(['python', 'training/forgetmi_partial_e30.py',
        '--config', CFG_BASE, '--seed', str(SEED), '--fresh',
        '--override', arg(ovr)], f'TRAIN {RID}')

elif METHOD in ('ng', 'cf'):
    # NegGrad+ / CF-k — scripts/unlearn_baselines.py import build_dataset cua
    # forgetmi_partial, nen cung phai qua launcher forgetmi_partial_e30.
    ovr = dict(COMMON)
    sh(['python', 'training/forgetmi_partial_e30.py', 'scripts/unlearn_baselines.py',
        '--config', CFG_BASE, '--method', {'ng': 'neggrad', 'cf': 'cfk'}[METHOD],
        '--seed', str(SEED), '--epochs', str(EPOCHS),
        '--override', arg(ovr)], f'TRAIN {RID}')

else:   # ref — OG va GOLD la checkpoint co san, chi do lai tren tap test moi
    for label, path in [('og', BASE), ('gold', GOLD)]:
        ovr = dict(COMMON); ovr['id'] = f'{label}_{DATASET}_{PCT}per_e30'
        sh(['python', 'training/forgetmi_e30.py', 'training/forgetmi_eval_only.py',
            '--config', CFG_P3, '--seed', str(SEED),
            '--label', f'{label}_{DATASET}_{PCT}per_e30',
            '--model_type', 'pretrained', '--model_path', path,
            '--method', 'reference', '--override', arg(ovr)], f'EVAL {label} {PCT}%')

print('\nxong Cell 3.')

In [ ]:
# Cell 4: BANG KET QUA + TU KIEM
import os
import pandas as pd
pd.set_option('display.width', 250)

COLS = ['id', 'checkpoint_kind', 'selected_epoch', 'Forget_AUC', 'Forget_Macro_F1',
        'Test_AUC', 'Test_Macro_F1', 'MIA', 'MIA_paper', 'forget_ce', 'test_ce',
        '1_minus_Sim', 'trainable_params', 'total_optimizer_steps', 'core_seconds']

print(f'===== {RESULTS} =====')
row = None
if os.path.exists(RESULTS):
    d = pd.read_csv(RESULTS)
    print(d[[c for c in COLS if c in d.columns]].to_string(index=False))
    if 'checkpoint_kind' in d.columns:
        _l = d[d['checkpoint_kind'] == 'last']
        if len(_l):
            row = _l.iloc[-1]
    elif len(d):
        row = d.iloc[-1]
else:
    print('CHUA CO FILE -> Cell 3 that bai, doc lai log ben tren')

print('\n===== TU KIEM =====')
if row is None:
    print('  khong co hang de kiem')
else:
    if METHOD == 'p3' and 'trainable_params' in row:
        n = int(row['trainable_params'])
        print(f'  tham so         {n:,}  ' +
              ('DUNG' if n == EXP_PARAMS else
               f'*** SAI: cho doi {EXP_PARAMS:,} -> CFG chua vao, BO ket qua ***'))
    if METHOD in ('p3', 'fmi') and 'total_optimizer_steps' in row:
        st = int(row['total_optimizer_steps'])
        print(f'  so lan cap nhat {st}  ' +
              ('DUNG' if st == EPOCHS else f'*** SAI: cho doi {EPOCHS} ***'))
    if 'checkpoint_kind' in row:
        print(f"  checkpoint      {row['checkpoint_kind']}  " +
              ("DUNG (E30)" if row['checkpoint_kind'] == 'last'
               else "*** chi bao cao hang 'last' ***"))

print('\n===== CHOT CHAN PHAN HOACH =====')
print('  Log Cell 3 PHAI co ba dong sau. Thieu bat ky dong nao -> ket qua KHONG dung')
print('  giao thuc E30, dung dua vao bang:')
print('    [split-e30] retain=6010/5810/5537 ... test_final=531 ... khong cat r_heldout/sel')
print('    adv_e30: PHAN HOACH THEO CODE GOC   (chi voi job p3_* va ref_*)')
print('    skip_selection=1 -> BO S_val moi epoch   (chi voi job p3_*)')
print('  Voi fmi_/ng_/cf_ thi dong can tim la:')
print('    Split E30 (bam code goc): train/retain=6010 test=531 ...')

print('\n===== FILE DE TAI VE =====')
for f in sorted(os.listdir('/kaggle/working')):
    if f.startswith(('results_e30', 'perepoch_')):
        print('   /kaggle/working/' + f)